# 03 · Detection + species classification

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/microsoft/SPARROW-Playground/blob/main/notebooks/03_detection_classification.ipynb)

A pipeline runs a detector, crops each box, and classifies the crop. Here MegaDetector V6
finds the animal and the AI4G Amazon classifier names it.

In [ ]:
# Setup: install sparrow-engine if it is missing (for example on Colab).
# For an NVIDIA GPU, install `sparrow-engine-gpu` instead (same Python API).
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("sparrow_engine") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "sparrow-engine[video]>=0.1.31"])

import sparrow_engine as spe

print("sparrow-engine", spe.__version__, "| device:", spe.active_device())

In [ ]:
# Demo data: downloaded once into ./data and checked against a published checksum.
import hashlib
import os
import shutil
import urllib.request
import zipfile
from pathlib import Path

DATA = Path(os.environ.get("SPP_DATA_DIR", "data")).resolve()
ASSETS = {
    # Pytorch-Wildlife demo assets, CC-BY-4.0, https://zenodo.org/records/15376499
    "demo_data_det.zip": (
        "https://zenodo.org/records/15376499/files/demo_data_det.zip?download=1",
        "md5", "373e15c8e7c8894379a82086a9177763",
    ),
    "demo_data_base.zip": (
        "https://zenodo.org/records/15376499/files/demo_data_base.zip?download=1",
        "md5", "f89b9b76c753d90e4296d62d71cf1732",
    ),
    # "Yellow-Throated Vireo Song" by SwampVids, CC-BY-4.0, Wikimedia Commons
    "vireo.wav": (
        "https://upload.wikimedia.org/wikipedia/commons/4/4f/"
        "Vireo_flavifrons_-_Yellow-Throated_Vireo_Song.wav",
        "sha1", "2b472209c1b18bfc0c225975838e3a34c1e9887a",
    ),
}


def fetch(name):
    """Download an asset (once), verify it, and unzip archives into DATA."""
    url, algo, digest = ASSETS[name]
    target = DATA / name
    DATA.mkdir(parents=True, exist_ok=True)
    if not target.exists():
        print(f"downloading {name} ...")
        part = target.with_name(target.name + ".part")
        req = urllib.request.Request(url, headers={"User-Agent": "sparrow-playground"})
        with urllib.request.urlopen(req) as r, open(part, "wb") as f:
            shutil.copyfileobj(r, f)
        part.rename(target)
    h = hashlib.new(algo)
    with open(target, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    if h.hexdigest() != digest:
        target.unlink()
        raise RuntimeError(f"{name}: checksum mismatch, deleted; run the cell again")
    marker = DATA / f".{name}.extracted"
    if name.endswith(".zip") and not marker.exists():
        with zipfile.ZipFile(target) as z:
            z.extractall(DATA)
        marker.touch()
    return target

In [ ]:
fetch("demo_data_base.zip")  # about 95 MB
images = [DATA / "demo_data/imgs/10050028_1.JPG", DATA / "demo_data/imgs/10050028_2.JPG"]

In [ ]:
# Models are installed into SPARROW_ENGINE_MODEL_DIR (default ~/.sparrow-engine/models).
# Already-installed models are skipped. Inference never downloads anything by itself.
for model_id in ["MDV6-yolov10-c", "AI4G-Amazon-V2"]:
    spe.download_model(model_id)

In [ ]:
results = spe.pipeline(images, detector="MDV6-yolov10-c", classifier="AI4G-Amazon-V2",
                       threshold=0.3)
for r in results:
    for d in r.detections:
        print(Path(r.path).name, d.detection.label, round(d.detection.confidence, 2),
              "->", d.classification.label, round(d.classification.confidence, 2))

In [ ]:
from IPython.display import Image, display

for png in spe.visualize(results, show_labels=True):
    display(Image(data=png, width=640))

## Why crop first?

Classifying the whole frame mixes the animal with the background. Compare the
whole-image prediction with the pipeline above:

In [ ]:
for r in spe.classify(images, model="AI4G-Amazon-V2", top_k=3):
    print(Path(r.path).name, [(c.label, round(c.confidence, 2)) for c in r.classifications])

## Other regional classifiers

| Classifier | Region | Licence |
|---|---|---|
| `AI4G-Amazon-V2` | Amazon rainforest | MIT |
| `AI4G-Serengeti` | Serengeti | MIT |
| `Deepfaune-Europe` | Europe | CC-BY-SA-4.0 |
| `Deepfaune-New-England` | New England | CC-BY-NC-SA-4.0 |
| `terai-nepal` | Terai, Nepal | MIT |
| `queensland` | Queensland | CC-BY-4.0 |

Notebook 01 lists the rest.